# Supply Forecasting
Eksperimen per wilayah, horizon 1/3/6 bulan. Membandingkan Seasonal Naive dan SARIMA jika statsmodels tersedia. 
Desain ini adalah rancangan eksperimen berdasarkan panel bulanan yang telah diaudit;

In [1]:
# 1. Cek environment — jalankan dengan kernel .venv project
import sys, importlib.util
import pandas as pd
import numpy as np

print("Python:", sys.executable)
print("pandas:", pd.__version__, "| numpy:", np.__version__)
HAS_STATSMODELS = importlib.util.find_spec("statsmodels") is not None
print("statsmodels tersedia:", HAS_STATSMODELS)
if HAS_STATSMODELS:
    import statsmodels
    print("statsmodels:", statsmodels.__version__)
else:
    print("SARIMA akan dilewati; baseline Seasonal Naive tetap dapat dijalankan.")


Python: d:\WIENY BISMILLAH S2\food_supply_chain\.venv\Scripts\python.exe
pandas: 3.0.6 | numpy: 2.5.3
statsmodels tersedia: True
statsmodels: 0.15.0


In [2]:
# 2. Baca dataset supply bulanan

from pathlib import Path
PROJECT_DIR = Path.cwd()
SUPPLY_PATH = PROJECT_DIR / "outputs" / "monthly_supply.csv"
if not SUPPLY_PATH.exists():
    raise FileNotFoundError(f"{SUPPLY_PATH} tidak ditemukan. Jalankan notebook dari root project")

df = pd.read_csv(SUPPLY_PATH)
required = {"region_standard", "period", "production_ton"}
if not required.issubset(df.columns):
    raise ValueError(f"Kolom wajib: {sorted(required)}")

df["period"] = pd.PeriodIndex(df["period"].astype(str), freq="M")
df["production_ton"] = pd.to_numeric(df["production_ton"], errors="coerce")
df = df.sort_values(["region_standard", "period"])

print("Rows:", len(df), "| regions:", df.region_standard.nunique)
print("Period:", df.period.min(), "-", df.period.max())
print("Missing:", df.production_ton.isna().sum(), "| zeo:", (df.production_ton == 0).sum())

display(df.head())

Rows: 2736 | regions: <bound method IndexOpsMixin.nunique of 0       Kabupaten Bangkalan
1       Kabupaten Bangkalan
2       Kabupaten Bangkalan
3       Kabupaten Bangkalan
4       Kabupaten Bangkalan
               ...         
2731          Kota Surabaya
2732          Kota Surabaya
2733          Kota Surabaya
2734          Kota Surabaya
2735          Kota Surabaya
Name: region_standard, Length: 2736, dtype: str>
Period: 2020-01 - 2025-12
Missing: 0 | zeo: 145


,region_id,region_standard,year,month,period,production_ton,rainfall_mm_month,temp_avg_c,humidity_pct,solar_mj,source_type,production_unit
0,KAB_BANGKALAN,Kabupaten Bangkalan,2020,1,2020-01,260.04,345.3,27.96,83.50,18.75,historical_observation,ton
1,KAB_BANGKALAN,Kabupaten Bangkalan,2020,2,2020-02,1407.42,435.3,27.46,85.92,18.56,historical_observation,ton
2,KAB_BANGKALAN,Kabupaten Bangkalan,2020,3,2020-03,12772.31,284.6,27.86,84.00,18.56,historical_observation,ton
3,KAB_BANGKALAN,Kabupaten Bangkalan,2020,4,2020-04,45623.93,329.7,28.23,82.48,17.77,historical_observation,ton
4,KAB_BANGKALAN,Kabupaten Bangkalan,2020,5,2020-05,8237.75,212.7,28.34,82.71,17.16,historical_observation,ton


# Rancangan evaluasi temporal
Data pelatihan bertambah seiring waktu (expanding window). Titik asal prediksi berjalan per bulan 2025; pada setiap titik, model membuat ramalan sampai 6 bulan. Hanya target yang jatuh pada 2025 yang dinilai. Ini mencegah pelatihan menggunakan observasi setelah titik asal prediksi.

Seasonal Naive memiliki nilai 12 bulan sebelumnya untuk bulan target yang sama. SARIMA memakai spesifikasi awal tetap (1,1,1) x (1,0,0,12) agar benchmark dapat direplikasi

In [3]:
# 4. Bentuk seri bbulanan lengkap per wilayah
series_by_region = {}
for region, part in df.groupby("region_standard"):
    part = part.drop_duplicates("period").set_index("period").sort_index()
    idx = pd.period_range(part.index.min(), part.index.max(), freq="M")
    series_by_region[region] = part["production_ton"].reindex(idx).astype(float)

print("Regions:", len(series_by_region))
print("Missing months total:", sum(s.isna().sum() for s in series_by_region.values()))

Regions: 38
Missing months total: 0


In [4]:
# Jalankan benchmark
from sklearn.metrics import  mean_absolute_error, mean_squared_error

HORIZONS = (1, 3, 6)
ORIGINS = pd.period_range("2024-12", "2025-12", freq="M")
rows = []

for region, y in series_by_region.items():
    for origin in ORIGINS:
        train = y.loc[:origin].dropna()
        if len(train) < 24:
            continue

        # Seasonal NAive
        for h in HORIZONS:
            target = origin + h
            source = target - 12
            if target.year != 2025 or target not in y.index or source not in y.index:
                continue
            actual, pred = y.loc[target], y.loc[source]

            if pd.notna(actual) and pd.notna(pred):
                rows.append(dict(region=region, origin=str(origin), target=str(target), horizon=h, model="SeasonalNaive", actual_ton=float(actual), forecast_ton=(pred)))

        # Fit once per region and origin, forecast up to six steps
        if HAS_STATSMODELS:
            try:
                from statsmodels.tsa.statespace.sarimax import SARIMAX
                fitted = SARIMAX(
                    train, order=(1,1,1), seasonal_order=(1,0,0,12), trend="n", enforce_stationarity=False, enforce_invertibility=False
                ).fit(disp=False, maxiter=100)
                fc = fitted.forecast(steps=6)
                for h in HORIZONS:
                    target = origin + h
                    if target.year != 2025 or target not in y.index:
                        continue
                    actual = y.loc[target]
                    if pd.notna(actual):
                        rows.append(dict(region=region, origin=str(origin), target=str(target), horizon=h, model="SARIMA_111_10112", actual_ton=float(actual), forecast_ton=float(fc.iloc[h-1])))
            except Exception as exc:
                print(f"SARIMA gagal untuk {region} pada {origin}: {exc}")

predictions = pd.DataFrame(rows)
if predictions.empty:
    raise RuntimeError("Tidak ada prediksi. Periksa path, data, dan cell sebelumnya")
print("Predictions:", len(predictions))
display(predictions.head())

d:\WIENY BISMILLAH S2\food_supply_chain\.venv\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:737: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(
d:\WIENY BISMILLAH S2\food_supply_chain\.venv\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:737: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(


Predictions: 2204


,region,origin,target,horizon,model,actual_ton,forecast_ton
0,Kabupaten Bangkalan,2024-12,2025-01,1,SeasonalNaive,981.05,0.000000
1,Kabupaten Bangkalan,2024-12,2025-03,3,SeasonalNaive,49059.63,31845.170000
2,Kabupaten Bangkalan,2024-12,2025-06,6,SeasonalNaive,15118.12,2863.980000
3,Kabupaten Bangkalan,2024-12,2025-01,1,SARIMA_111_10112,981.05,3173.228559
4,Kabupaten Bangkalan,2024-12,2025-03,3,SARIMA_111_10112,49059.63,23787.564446


In [5]:
# 6. Evaluasi MAE dan RMSE
predictions["abs_error"] = (predictions.actual_ton - predictions.forecast_ton).abs()
predictions["squared_error"] = (predictions.actual_ton - predictions.forecast_ton) ** 2

metrics_overall = predictions.groupby(["model", "horizon"], as_index=False).agg(
    MAE=("abs_error", "mean"),
    RMSE=("squared_error", lambda x: float(np.sqrt(x.mean()))),
    n_predictions=("abs_error", "size")
)
metrics_by_region = predictions.groupby(["model", "horizon", "region"], as_index=False).agg(
    MAE=("abs_error", "mean"),
    RMSE=("squared_error", lambda x: float(np.sqrt(x.mean()))),
    n_predictions=("abs_error", "size")
)
display(metrics_overall.sort_values(["horizon", "MAE"]))
display(metrics_by_region.head(15))

,model,horizon,MAE,RMSE,n_predictions
0,SARIMA_111_10112,1,5016.056141,10217.657055,456
3,SeasonalNaive,1,5037.510833,10189.727694,456
1,SARIMA_111_10112,3,5099.303008,10612.497533,380
4,SeasonalNaive,3,5394.969421,10789.877052,380
2,SARIMA_111_10112,6,3656.693170,8168.872002,266
5,SeasonalNaive,6,3827.884511,8597.032163,266


,model,horizon,region,MAE,RMSE,n_predictions
0,SARIMA_111_10112,1,Kabupaten Bangkalan,6821.191088,9164.791325,12
1,SARIMA_111_10112,1,Kabupaten Banyuwangi,4627.501157,5925.689927,12
2,SARIMA_111_10112,1,Kabupaten Blitar,3658.581712,6472.255346,12
3,SARIMA_111_10112,1,Kabupaten Bojonegoro,15422.440007,25268.454781,12
4,SARIMA_111_10112,1,Kabupaten Bondowoso,2284.941194,2907.496442,12
5,SARIMA_111_10112,1,Kabupaten Gresik,11080.717331,14516.296657,12
6,SARIMA_111_10112,1,Kabupaten Jember,5009.812669,6261.776523,12
7,SARIMA_111_10112,1,Kabupaten Jombang,6439.165078,7430.585523,12
8,SARIMA_111_10112,1,Kabupaten Kediri,2710.116454,3954.100109,12
9,SARIMA_111_10112,1,Kabupaten Lamongan,20606.489911,26026.505769,12


In [6]:
# 7. Simpan keluaran
OUT = PROJECT_DIR / "outputs" / "forecasting_benchmark"
OUT.mkdir(parents=True, exist_ok=True)
predictions.to_csv(OUT / "supply_predictions.csv", index=False)
metrics_overall.to_csv(OUT / "supply_metrics_overall.csv", index=False)
metrics_by_region.to_csv(OUT / "supply_metrics_by_region.csv", index=False)
print("Hasil tersimpan di:", OUT.resolve())


Hasil tersimpan di: D:\WIENY BISMILLAH S2\food_supply_chain\outputs\forecasting_benchmark


8. Batasan interpretasi
- Nilai produksi nol dipertahankan sesuai sumber; perlu verifikasi dan analisis sensitivitas, bukan imputasi otomatis.
- Spesifikasi SARIMA adalah konfigurasi benchmark awal, bukan hasil optimasi parameter.
- Belum ada variabel iklim pada notebook ini. Uji iklim dilakukan terpisah setelah baseline; nilai iklim masa depan tidak boleh digunakan jika belum tersedia pada waktu prediksi.
- Demand belum dimodelkan sebagai deret konsumsi bulanan karena nilai bulanan saat ini merupakan alokasi rata-rata dari total tahunan.
- Bandingkan MAE/RMSE pada horizon dan cakupan wilayah yang sama. Jangan memilih model hanya karena lebih kompleks.

In [7]:
import pandas as pd
import numpy as np

pred = pd.read_csv(
    "outputs/forecasting_benchmark/supply_predictions.csv"
)

print("Kolom tersedia:", pred.columns.tolist())
display(pred.head())


Kolom tersedia: ['region', 'origin', 'target', 'horizon', 'model', 'actual_ton', 'forecast_ton', 'abs_error', 'squared_error']


,region,origin,target,horizon,model,actual_ton,forecast_ton,abs_error,squared_error
0,Kabupaten Bangkalan,2024-12,2025-01,1,SeasonalNaive,981.05,0.000000,981.050000,9.624591e+05
1,Kabupaten Bangkalan,2024-12,2025-03,3,SeasonalNaive,49059.63,31845.170000,17214.460000,2.963376e+08
2,Kabupaten Bangkalan,2024-12,2025-06,6,SeasonalNaive,15118.12,2863.980000,12254.140000,1.501639e+08
3,Kabupaten Bangkalan,2024-12,2025-01,1,SARIMA_111_10112,981.05,3173.228559,2192.178559,4.805647e+06
4,Kabupaten Bangkalan,2024-12,2025-03,3,SARIMA_111_10112,49059.63,23787.564446,25272.065554,6.386773e+08


In [8]:

import pandas as pd
import numpy as np

# Load hasil prediksi
pred = pd.read_csv(
    "outputs/forecasting_benchmark/supply_predictions.csv"
)

# Pastikan tipe data benar
pred["origin"] = pd.to_datetime(pred["origin"])
pred["target"] = pd.to_datetime(pred["target"])

# Tambahkan error bertanda:
# positif = model terlalu tinggi (overforecast)
# negatif = model terlalu rendah (underforecast)
pred["error"] = pred["forecast_ton"] - pred["actual_ton"]

# ==================================================
# A. PERFORMA MODEL PER WILAYAH DAN HORIZON
# ==================================================
by_region = (
    pred.groupby(["region", "horizon", "model"])
    .agg(
        MAE=("abs_error", "mean"),
        RMSE=("squared_error", lambda x: np.sqrt(x.mean())),
        Bias=("error", "mean"),
        n_predictions=("abs_error", "count")
    )
    .reset_index()
)

print("=== Performa model per wilayah dan horizon ===")
display(by_region.sort_values(
    ["horizon", "MAE"]
).head(20))

# Simpan hasil
by_region.to_csv(
    "outputs/forecasting_benchmark/metrics_by_region.csv",
    index=False
)

# ==================================================
# B. PERBANDINGAN SARIMA VS SEASONAL NAIVE
#    PADA WILAYAH YANG SAMA
# ==================================================
comparison = by_region.pivot_table(
    index=["region", "horizon"],
    columns="model",
    values=["MAE", "RMSE"],
    aggfunc="first"
)

comparison.columns = [
    f"{metric}_{model}" for metric, model in comparison.columns
]
comparison = comparison.reset_index()

comparison["MAE_improvement_pct"] = (
    (comparison["MAE_SeasonalNaive"] -
     comparison["MAE_SARIMA_111_10112"])
    / comparison["MAE_SeasonalNaive"] * 100
)

comparison["SARIMA_better_MAE"] = (
    comparison["MAE_improvement_pct"] > 0
)

print("=== Persentase wilayah yang lebih baik dengan SARIMA ===")
display(
    comparison.groupby("horizon")
    .agg(
        total_regions=("region", "nunique"),
        regions_SARIMA_better=("SARIMA_better_MAE", "sum"),
        average_improvement_pct=("MAE_improvement_pct", "mean")
    )
    .reset_index()
)

comparison.to_csv(
    "outputs/forecasting_benchmark/model_comparison_by_region.csv",
    index=False
)

# ==================================================
# C. 10 PREDIKSI DENGAN ERROR TERBESAR
# ==================================================
print("=== 10 error prediksi terbesar ===")
display(
    pred.sort_values("abs_error", ascending=False)[
        [
            "region", "origin", "target", "horizon",
            "model", "actual_ton", "forecast_ton",
            "error", "abs_error"
        ]
    ].head(10)
)

# ==================================================
# D. ERROR BERDASARKAN PERIODE TARGET
# ==================================================
pred["target_month"] = pred["target"].dt.month

by_month = (
    pred.groupby(["target_month", "horizon", "model"])
    .agg(
        MAE=("abs_error", "mean"),
        RMSE=("squared_error", lambda x: np.sqrt(x.mean())),
        n_predictions=("abs_error", "count")
    )
    .reset_index()
)

print("=== Error berdasarkan bulan target ===")
display(by_month.sort_values(["horizon", "MAE"]))

by_month.to_csv(
    "outputs/forecasting_benchmark/metrics_by_month.csv",
    index=False
)

# ==================================================
# E. CEK ERROR PADA PRODUKSI AKTUAL NOL
# ==================================================
zero_actual = (
    pred[pred["actual_ton"] == 0]
    .groupby(["horizon", "model"])
    .agg(
        n_zero_actual=("actual_ton", "count"),
        mean_abs_error=("abs_error", "mean")
    )
    .reset_index()
)

print("=== Error ketika produksi aktual nol ===")
display(zero_actual)

print("Analisis selesai.")


=== Performa model per wilayah dan horizon ===


,region,horizon,model,MAE,RMSE,Bias,n_predictions
175,Kota Batu,1,SeasonalNaive,72.755000,96.274338,30.530000,12
205,Kota Mojokerto,1,SeasonalNaive,83.304167,113.762039,17.515833,12
204,Kota Mojokerto,1,SARIMA_111_10112,83.681399,97.199769,16.640762,12
174,Kota Batu,1,SARIMA_111_10112,93.909808,110.216604,1.917380,12
198,Kota Malang,1,SARIMA_111_10112,154.097028,166.600852,102.952309,12
217,Kota Probolinggo,1,SeasonalNaive,165.575833,292.296049,-76.887500,12
216,Kota Probolinggo,1,SARIMA_111_10112,167.277812,306.825155,-61.462491,12
199,Kota Malang,1,SeasonalNaive,192.377500,227.767455,108.445833,12
192,Kota Madiun,1,SARIMA_111_10112,208.654566,300.863688,16.031223,12
181,Kota Blitar,1,SeasonalNaive,239.195833,428.843748,-112.874167,12


=== Persentase wilayah yang lebih baik dengan SARIMA ===


,horizon,total_regions,regions_SARIMA_better,average_improvement_pct
0,1,38,18,0.567074
1,3,38,24,5.807428
2,6,38,20,-113.069507


=== 10 error prediksi terbesar ===


,region,origin,target,horizon,model,actual_ton,forecast_ton,error,abs_error
1533,Kabupaten Trenggalek,2025-04-01,2025-07-01,3,SeasonalNaive,102299.60,4151.930000,-98147.670000,98147.670000
1516,Kabupaten Trenggalek,2025-01-01,2025-07-01,6,SeasonalNaive,102299.60,4151.930000,-98147.670000,98147.670000
1544,Kabupaten Trenggalek,2025-06-01,2025-07-01,1,SeasonalNaive,102299.60,4151.930000,-98147.670000,98147.670000
1519,Kabupaten Trenggalek,2025-01-01,2025-07-01,6,SARIMA_111_10112,102299.60,4320.342031,-97979.257969,97979.257969
1547,Kabupaten Trenggalek,2025-06-01,2025-07-01,1,SARIMA_111_10112,102299.60,4721.459462,-97578.140538,97578.140538
1536,Kabupaten Trenggalek,2025-04-01,2025-07-01,3,SARIMA_111_10112,102299.60,5428.950721,-96870.649279,96870.649279
189,Kabupaten Bojonegoro,2025-02-01,2025-03-01,1,SARIMA_111_10112,137540.98,58135.435486,-79405.544514,79405.544514
178,Kabupaten Bojonegoro,2024-12-01,2025-03-01,3,SARIMA_111_10112,137540.98,59231.952372,-78309.027628,78309.027628
186,Kabupaten Bojonegoro,2025-02-01,2025-03-01,1,SeasonalNaive,137540.98,70525.130000,-67015.850000,67015.850000
175,Kabupaten Bojonegoro,2024-12-01,2025-03-01,3,SeasonalNaive,137540.98,70525.130000,-67015.850000,67015.850000


=== Error berdasarkan bulan target ===


,target_month,horizon,model,MAE,RMSE,n_predictions
52,12,1,SARIMA_111_10112,1834.593227,2898.523609,38
53,12,1,SeasonalNaive,2008.640263,3251.393418,38
1,1,1,SeasonalNaive,2153.352895,4447.921336,38
0,1,1,SARIMA_111_10112,2663.455685,4045.463466,38
41,10,1,SeasonalNaive,3054.627105,5778.608592,38
40,10,1,SARIMA_111_10112,3124.598335,5083.726759,38
46,11,1,SARIMA_111_10112,3131.464366,5047.835022,38
29,8,1,SeasonalNaive,3325.536316,5827.747525,38
35,9,1,SeasonalNaive,3675.200526,6587.847618,38
34,9,1,SARIMA_111_10112,3675.308305,6018.394141,38


=== Error ketika produksi aktual nol ===


,horizon,model,n_zero_actual,mean_abs_error
0,1,SARIMA_111_10112,29,897.655164
1,1,SeasonalNaive,29,68.615517
2,3,SARIMA_111_10112,24,957.385950
3,3,SeasonalNaive,24,49.455000
4,6,SARIMA_111_10112,22,1034.364538
5,6,SeasonalNaive,22,50.218636


Analisis selesai.


In [9]:

import pandas as pd
import numpy as np

supply = pd.read_csv("outputs/monthly_supply.csv")
supply["period"] = pd.to_datetime(supply["period"])

# Periksa dua wilayah dengan error terbesar
regions_to_check = [
    "Kabupaten Trenggalek",
    "Kabupaten Bojonegoro"
]

for region in regions_to_check:
    print(f"\n=== {region} ===")

    subset = supply[
        supply["region_standard"] == region
    ].sort_values("period")

    subset["previous_month"] = subset["production_ton"].shift(1)
    subset["change_ton"] = (
        subset["production_ton"] - subset["previous_month"]
    )
    subset["change_pct"] = (
        subset["change_ton"] /
        subset["previous_month"].replace(0, np.nan) * 100
    )

    display(
        subset[
            ["period", "production_ton",
             "previous_month", "change_ton", "change_pct"]
        ].tail(24)
    )

# Periksa apakah nilai ekstrem hanya terjadi pada dua wilayah
print("\n=== 10 produksi bulanan terbesar pada 2025 ===")
display(
    supply[supply["period"].dt.year == 2025]
    .nlargest(10, "production_ton")[
        ["region_standard", "period", "production_ton"]
    ]
)




=== Kabupaten Trenggalek ===


,period,production_ton,previous_month,change_ton,change_pct
1920,2024-01-01,1126.18,2776.44,-1650.26,-59.437985
1921,2024-02-01,231.97,1126.18,-894.21,-79.402049
1922,2024-03-01,3749.99,231.97,3518.02,1516.584041
1923,2024-04-01,16140.19,3749.99,12390.20,330.406214
1924,2024-05-01,10103.66,16140.19,-6036.53,-37.400613
1925,2024-06-01,3525.51,10103.66,-6578.15,-65.106605
1926,2024-07-01,4151.93,3525.51,626.42,17.768209
1927,2024-08-01,11990.38,4151.93,7838.45,188.790514
1928,2024-09-01,2571.80,11990.38,-9418.58,-78.551138
1929,2024-10-01,3086.34,2571.80,514.54,20.006999



=== Kabupaten Bojonegoro ===


,period,production_ton,previous_month,change_ton,change_pct
264,2024-01-01,9144.98,7845.98,1299.00,16.556249
265,2024-02-01,23973.16,9144.98,14828.18,162.145571
266,2024-03-01,70525.13,23973.16,46551.97,194.183704
267,2024-04-01,103826.12,70525.13,33300.99,47.218616
268,2024-05-01,29679.17,103826.12,-74146.95,-71.414544
269,2024-06-01,47774.05,29679.17,18094.88,60.968282
270,2024-07-01,42024.40,47774.05,-5749.65,-12.035090
271,2024-08-01,20039.93,42024.40,-21984.47,-52.313584
272,2024-09-01,17279.50,20039.93,-2760.43,-13.774649
273,2024-10-01,17896.94,17279.50,617.44,3.573252



=== 10 produksi bulanan terbesar pada 2025 ===


,region_standard,period,production_ton
710,Kabupaten Lamongan,2025-03-01,144391.63
278,Kabupaten Bojonegoro,2025-03-01,137540.98
1938,Kabupaten Trenggalek,2025-07-01,102299.60
495,Kabupaten Jember,2025-04-01,94951.31
279,Kabupaten Bojonegoro,2025-04-01,83387.26
567,Kabupaten Jombang,2025-04-01,75436.73
2007,Kabupaten Tuban,2025-04-01,70809.03
422,Kabupaten Gresik,2025-03-01,70655.86
1646,Kabupaten Sampang,2025-03-01,65117.54
1214,Kabupaten Ngawi,2025-03-01,64973.64


In [10]:

import pandas as pd
import numpy as np

pred = pd.read_csv(
    "outputs/forecasting_benchmark/supply_predictions.csv"
)

keys = ["region", "origin", "target", "horizon"]

paired = pred.pivot_table(
    index=keys,
    columns="model",
    values=["actual_ton", "abs_error", "squared_error"],
    aggfunc="first"
).dropna()

# Hitung ulang MAE dan RMSE pada pasangan observasi identik
for model in ["SARIMA_111_10112", "SeasonalNaive"]:
    pass

paired_metrics = []

for horizon, group in paired.groupby(level="horizon"):
    row = {"horizon": horizon}

    for model in ["SARIMA_111_10112", "SeasonalNaive"]:
        errors = group[("abs_error", model)]
        squared = group[("squared_error", model)]

        row[f"MAE_{model}"] = errors.mean()
        row[f"RMSE_{model}"] = np.sqrt(squared.mean())

    row["SARIMA_MAE_improvement_pct"] = (
        (row["MAE_SeasonalNaive"] -
         row["MAE_SARIMA_111_10112"])
        / row["MAE_SeasonalNaive"] * 100
    )

    paired_metrics.append(row)

paired_metrics = pd.DataFrame(paired_metrics)
display(paired_metrics)

# Kemenangan SARIMA per wilayah, berdasarkan pasangan yang sama
region_metrics = []

for (region, horizon), group in paired.groupby(
    level=["region", "horizon"]
):
    sarima_mae = group[("abs_error", "SARIMA_111_10112")].mean()
    naive_mae = group[("abs_error", "SeasonalNaive")].mean()

    region_metrics.append({
        "region": region,
        "horizon": horizon,
        "SARIMA_MAE": sarima_mae,
        "Naive_MAE": naive_mae,
        "improvement_pct": (
            (naive_mae - sarima_mae) / naive_mae * 100
            if naive_mae != 0 else np.nan
        ),
        "SARIMA_better": sarima_mae < naive_mae
    })

region_metrics = pd.DataFrame(region_metrics)

display(
    region_metrics.groupby("horizon").agg(
        total_regions=("region", "nunique"),
        regions_SARIMA_better=("SARIMA_better", "sum"),
        mean_improvement_pct=("improvement_pct", "mean"),
        median_improvement_pct=("improvement_pct", "median")
    ).reset_index()
)

region_metrics.to_csv(
    "outputs/forecasting_benchmark/paired_region_comparison.csv",
    index=False
)


,horizon,MAE_SARIMA_111_10112,RMSE_SARIMA_111_10112,MAE_SeasonalNaive,RMSE_SeasonalNaive,SARIMA_MAE_improvement_pct
0,1,5016.056141,10217.657055,5037.510833,10189.727694,0.425899
1,3,5099.303008,10612.497533,5394.969421,10789.877052,5.480409
2,6,3656.693170,8168.872002,3827.884511,8597.032163,4.472218


,horizon,total_regions,regions_SARIMA_better,mean_improvement_pct,median_improvement_pct
0,1,38,18,0.567074,-0.504011
1,3,38,24,5.807428,5.135615
2,6,38,20,-113.069507,6.173357


In [11]:

import pandas as pd
import numpy as np

supply = pd.read_csv("outputs/monthly_supply.csv")
supply["period"] = pd.to_datetime(supply["period"])

supply["year"] = supply["period"].dt.year
supply["month"] = supply["period"].dt.month

# Agregasi produksi seluruh kabupaten/kota
monthly_total = (
    supply.groupby(["year", "month"])["production_ton"]
    .sum()
    .reset_index()
)

# Rata-rata produksi untuk setiap bulan selama 2020-2025
monthly_pattern = (
    monthly_total.groupby("month")["production_ton"]
    .agg(["mean", "median", "std", "min", "max"])
    .reset_index()
)

monthly_pattern["month_name"] = monthly_pattern["month"].map({
    1: "Januari", 2: "Februari", 3: "Maret",
    4: "April", 5: "Mei", 6: "Juni",
    7: "Juli", 8: "Agustus", 9: "September",
    10: "Oktober", 11: "November", 12: "Desember"
})

display(monthly_pattern)

# Periksa pola tiap tahun
display(
    monthly_total.pivot(
        index="month",
        columns="year",
        values="production_ton"
    )
)

# Simpan
monthly_pattern.to_csv(
    "outputs/forecasting_benchmark/monthly_production_pattern.csv",
    index=False
)


,month,mean,median,std,min,max,month_name
0,1,1.779185e+05,170199.860,22801.823696,159965.55,220836.08,Januari
1,2,3.237909e+05,312228.890,140096.847081,179135.66,564864.35,Februari
2,3,1.085231e+06,1198491.290,253494.253041,763040.96,1320825.83,Maret
3,4,1.006988e+06,1004047.750,235513.399982,685517.49,1295711.95,April
4,5,4.084767e+05,381656.145,107632.285426,307442.64,572853.84,Mei
5,6,4.483450e+05,471885.760,72658.725003,348426.82,542818.37,Juni
6,7,6.140506e+05,633345.945,75463.831244,513882.26,693479.47,Juli
7,8,4.537132e+05,452718.000,60440.520918,370240.07,533598.35,Agustus
8,9,3.255618e+05,325682.770,48914.382474,265998.82,379459.67,September
9,10,2.970056e+05,282396.080,42253.742665,247663.57,357018.02,Oktober


year,2020,2021,2022,2023,2024,2025
month,,,,,,
1,166713.38,173686.34,162052.21,184257.56,159965.55,220836.08
2,199349.38,293189.35,374938.00,564864.35,179135.66,331268.43
3,764516.15,1266018.00,1320825.83,1218335.62,763040.96,1178646.96
4,1295711.95,958043.07,819381.66,685517.49,1233221.75,1050052.43
5,572853.84,307442.64,307630.67,361908.46,499620.93,401403.83
6,348426.82,468542.04,475229.48,542818.37,375133.56,479920.00
7,655875.57,693479.47,610816.32,533668.13,513882.26,676582.01
8,533598.35,412947.06,370240.07,429221.06,500057.68,476214.94
9,345921.40,279918.48,265998.82,305444.14,376628.51,379459.67


In [12]:

import pandas as pd
import numpy as np

supply = pd.read_csv("outputs/monthly_supply.csv")
supply["period"] = pd.to_datetime(supply["period"])

supply["year"] = supply["period"].dt.year
supply["month"] = supply["period"].dt.month

# Total produksi per tahun dan bulan
monthly = (
    supply.groupby(["year", "month"])["production_ton"]
    .sum()
    .reset_index()
)

# Total produksi tahunan
annual = (
    monthly.groupby("year")["production_ton"]
    .sum()
    .rename("annual_production")
)

# Produksi Maret-April
mar_apr = (
    monthly[monthly["month"].isin([3, 4])]
    .groupby("year")["production_ton"]
    .sum()
    .rename("mar_apr_production")
)

seasonality = pd.concat([annual, mar_apr], axis=1)
seasonality["mar_apr_share_pct"] = (
    seasonality["mar_apr_production"] /
    seasonality["annual_production"] * 100
)

display(seasonality.round(2))

# Pola rata-rata bulanan selama 6 tahun
monthly_pattern = (
    monthly.groupby("month")["production_ton"]
    .agg(["mean", "std"])
    .reset_index()
)

monthly_pattern["cv_pct"] = (
    monthly_pattern["std"] /
    monthly_pattern["mean"] * 100
)

display(monthly_pattern.round(2))

# Simpan hasil
seasonality.to_csv(
    "outputs/forecasting_benchmark/mar_apr_contribution.csv"
    , index=True
)

monthly_pattern.to_csv(
    "outputs/forecasting_benchmark/monthly_seasonality_statistics.csv",
    index=False
)


,annual_production,mar_apr_production,mar_apr_share_pct
year,,,
2020,5742176.67,2060228.10,35.88
2021,5652705.10,2224061.07,39.35
2022,5500801.88,2140207.49,38.91
2023,5607131.53,1903853.11,33.95
2024,5352936.04,1996262.71,37.29
2025,6148189.18,2228699.39,36.25


,month,mean,std,cv_pct
0,1,177918.52,22801.82,12.82
1,2,323790.86,140096.85,43.27
2,3,1085230.59,253494.25,23.36
3,4,1006988.06,235513.40,23.39
4,5,408476.73,107632.29,26.35
5,6,448345.04,72658.73,16.21
6,7,614050.63,75463.83,12.29
7,8,453713.19,60440.52,13.32
8,9,325561.84,48914.38,15.02
9,10,297005.58,42253.74,14.23


In [14]:

import pandas as pd

supply = pd.read_csv("outputs/monthly_supply.csv")
climate = pd.read_csv("data/jatim_climate_monthly (1).csv")

print("Kolom supply:")
print(supply.columns.tolist())

print("\nKolom climate:")
print(climate.columns.tolist())

print("\nContoh data climate:")
display(climate.head())

print("\nPeriode climate:")
print(climate["period"].min(), climate["period"].max())

print("\nJumlah wilayah climate:")
print(climate["region"].nunique())


Kolom supply:
['region_id', 'region_standard', 'year', 'month', 'period', 'production_ton', 'rainfall_mm_month', 'temp_avg_c', 'humidity_pct', 'solar_mj', 'source_type', 'production_unit']

Kolom climate:
['region', 'lat', 'lon', 'period', 'rainfall_mm_per_day', 'rainfall_mm_month', 'temp_avg_c', 'temp_max_c', 'temp_min_c', 'humidity_pct', 'solar_mj']

Contoh data climate:


,region,lat,lon,period,rainfall_mm_per_day,rainfall_mm_month,temp_avg_c,temp_max_c,temp_min_c,humidity_pct,solar_mj
0,Kab. Bangkalan,-7.0455,112.7447,2020-01,11.14,345.3,27.96,31.67,25.02,83.50,18.75
1,Kab. Bangkalan,-7.0455,112.7447,2020-02,15.01,435.3,27.46,30.71,24.95,85.92,18.56
2,Kab. Bangkalan,-7.0455,112.7447,2020-03,9.18,284.6,27.86,30.77,25.30,84.00,18.56
3,Kab. Bangkalan,-7.0455,112.7447,2020-04,10.99,329.7,28.23,30.99,25.31,82.48,17.77
4,Kab. Bangkalan,-7.0455,112.7447,2020-05,6.86,212.7,28.34,31.39,26.17,82.71,17.16



Periode climate:
2020-01 2026-10

Jumlah wilayah climate:
38


In [16]:

import pandas as pd
import numpy as np

supply = pd.read_csv("outputs/monthly_supply.csv")
climate = pd.read_csv("data/jatim_climate_monthly (1).csv")

# Samakan format periode
supply["period"] = supply["period"].astype(str)
climate["period"] = climate["period"].astype(str)

# Periksa duplikasi kunci wilayah-bulan
print("Duplikasi supply:",
      supply.duplicated(["region_standard", "period"]).sum())

print("Duplikasi climate:",
      climate.duplicated(["region", "period"]).sum())

# Gabungkan untuk memeriksa konsistensi nilai iklim
check = supply.merge(
    climate,
    left_on=["region_standard", "period"],
    right_on=["region", "period"],
    how="left",
    suffixes=("_supply", "_climate"),
    indicator=True
)

print("\nStatus pencocokan:")
print(check["_merge"].value_counts())

variables = [
    "rainfall_mm_month",
    "temp_avg_c",
    "humidity_pct",
    "solar_mj"
]

print("\nPerbandingan nilai iklim:")
for col in variables:
    a = check[f"{col}_supply"]
    b = check[f"{col}_climate"]

    valid = a.notna() & b.notna()
    diff = (a[valid] - b[valid]).abs()

    print(
        col,
        "| matched:", valid.sum(),
        "| missing:", (~valid).sum(),
        "| max_abs_diff:", diff.max() if len(diff) else None
    )

# Periksa kelengkapan seluruh variabel iklim
print("\nMissing values pada supply:")
print(supply[variables].isna().sum())

print("\nMissing values pada climate:")
climate_variables = [
    "rainfall_mm_month", "temp_avg_c", "temp_max_c",
    "temp_min_c", "humidity_pct", "solar_mj"
]
print(climate[climate_variables].isna().sum())


Duplikasi supply: 0
Duplikasi climate: 0

Status pencocokan:
_merge
left_only     2088
both           648
right_only       0
Name: count, dtype: int64

Perbandingan nilai iklim:
rainfall_mm_month | matched: 648 | missing: 2088 | max_abs_diff: 0.0
temp_avg_c | matched: 648 | missing: 2088 | max_abs_diff: 0.0
humidity_pct | matched: 648 | missing: 2088 | max_abs_diff: 0.0
solar_mj | matched: 648 | missing: 2088 | max_abs_diff: 0.0

Missing values pada supply:
rainfall_mm_month    0
temp_avg_c           0
humidity_pct         0
solar_mj             0
dtype: int64

Missing values pada climate:
rainfall_mm_month     76
temp_avg_c            76
temp_max_c            76
temp_min_c            76
humidity_pct          76
solar_mj             190
dtype: int64


In [17]:

# Periksa distribusi periode hasil pencocokan
print("Periode supply:")
print(supply.groupby(supply["period"].str[:4]).size())

print("\nPeriode climate:")
print(climate.groupby(climate["period"].str[:4]).size())

print("\nPeriode yang berhasil cocok:")
print(
    check[check["_merge"] == "both"]
    .groupby(check.loc[check["_merge"] == "both", "period"].str[:4])
    .size()
)

# Periksa nama wilayah yang tidak cocok
supply_regions = set(supply["region_standard"].dropna().unique())
climate_regions = set(climate["region"].dropna().unique())

print("\nWilayah hanya di supply:")
print(sorted(supply_regions - climate_regions))

print("\nWilayah hanya di climate:")
print(sorted(climate_regions - supply_regions))

# Periksa periode supply yang tidak cocok
print("\nContoh baris supply yang tidak cocok:")
display(
    check.loc[
        check["_merge"] == "left_only",
        ["region_standard", "period"]
    ].head(20)
)

# Periksa kelengkapan iklim menurut tahun
print("\nMissing values climate per tahun:")
print(
    climate.assign(year=climate["period"].str[:4])
    .groupby("year")[climate_variables]
    .apply(lambda x: x.isna().sum())
)


Periode supply:
period
2020    456
2021    456
2022    456
2023    456
2024    456
2025    456
dtype: int64

Periode climate:
period
2020    456
2021    456
2022    456
2023    456
2024    456
2025    456
2026    380
dtype: int64

Periode yang berhasil cocok:
period
2020    108
2021    108
2022    108
2023    108
2024    108
2025    108
dtype: int64

Wilayah hanya di supply:
['Kabupaten Bangkalan', 'Kabupaten Banyuwangi', 'Kabupaten Blitar', 'Kabupaten Bojonegoro', 'Kabupaten Bondowoso', 'Kabupaten Gresik', 'Kabupaten Jember', 'Kabupaten Jombang', 'Kabupaten Kediri', 'Kabupaten Lamongan', 'Kabupaten Lumajang', 'Kabupaten Madiun', 'Kabupaten Magetan', 'Kabupaten Malang', 'Kabupaten Mojokerto', 'Kabupaten Nganjuk', 'Kabupaten Ngawi', 'Kabupaten Pacitan', 'Kabupaten Pamekasan', 'Kabupaten Pasuruan', 'Kabupaten Ponorogo', 'Kabupaten Probolinggo', 'Kabupaten Sampang', 'Kabupaten Sidoarjo', 'Kabupaten Situbondo', 'Kabupaten Sumenep', 'Kabupaten Trenggalek', 'Kabupaten Tuban', 'Kabupaten Tulu

,region_standard,period
0,Kabupaten Bangkalan,2020-01
1,Kabupaten Bangkalan,2020-02
2,Kabupaten Bangkalan,2020-03
3,Kabupaten Bangkalan,2020-04
4,Kabupaten Bangkalan,2020-05
5,Kabupaten Bangkalan,2020-06
6,Kabupaten Bangkalan,2020-07
7,Kabupaten Bangkalan,2020-08
8,Kabupaten Bangkalan,2020-09
9,Kabupaten Bangkalan,2020-10



Missing values climate per tahun:
      rainfall_mm_month  temp_avg_c  temp_max_c  temp_min_c  humidity_pct  \
year                                                                        
2020                  0           0           0           0             0   
2021                  0           0           0           0             0   
2022                  0           0           0           0             0   
2023                  0           0           0           0             0   
2024                  0           0           0           0             0   
2025                  0           0           0           0             0   
2026                 76          76          76          76            76   

      solar_mj  
year            
2020         0  
2021         0  
2022         0  
2023         0  
2024         0  
2025         0  
2026       190  


In [18]:

print("Nama wilayah pada SUPPLY:")
print(sorted(supply["region_standard"].unique())[:15])

print("\nNama wilayah pada CLIMATE:")
print(sorted(climate["region"].unique())[:15])

print("\nContoh seluruh pasangan nama:")
print(
    supply[["region_standard"]]
    .drop_duplicates()
    .head(10)
    .to_string(index=False)
)

print(
    climate[["region"]]
    .drop_duplicates()
    .head(10)
    .to_string(index=False)
)


Nama wilayah pada SUPPLY:
['Kabupaten Bangkalan', 'Kabupaten Banyuwangi', 'Kabupaten Blitar', 'Kabupaten Bojonegoro', 'Kabupaten Bondowoso', 'Kabupaten Gresik', 'Kabupaten Jember', 'Kabupaten Jombang', 'Kabupaten Kediri', 'Kabupaten Lamongan', 'Kabupaten Lumajang', 'Kabupaten Madiun', 'Kabupaten Magetan', 'Kabupaten Malang', 'Kabupaten Mojokerto']

Nama wilayah pada CLIMATE:
['Kab. Bangkalan', 'Kab. Banyuwangi', 'Kab. Blitar', 'Kab. Bojonegoro', 'Kab. Bondowoso', 'Kab. Gresik', 'Kab. Jember', 'Kab. Jombang', 'Kab. Kediri', 'Kab. Lamongan', 'Kab. Lumajang', 'Kab. Madiun', 'Kab. Magetan', 'Kab. Malang', 'Kab. Mojokerto']

Contoh seluruh pasangan nama:
     region_standard
 Kabupaten Bangkalan
Kabupaten Banyuwangi
    Kabupaten Blitar
Kabupaten Bojonegoro
 Kabupaten Bondowoso
    Kabupaten Gresik
    Kabupaten Jember
   Kabupaten Jombang
    Kabupaten Kediri
  Kabupaten Lamongan
         region
 Kab. Bangkalan
Kab. Banyuwangi
    Kab. Blitar
Kab. Bojonegoro
 Kab. Bondowoso
    Kab. Gresik

In [20]:

import pandas as pd

supply = pd.read_csv("outputs/monthly_supply.csv")
climate = pd.read_csv("data/jatim_climate_monthly (1).csv")

def standardize_region(name):
    name = str(name).strip()

    name = name.replace("Kabupaten ", "Kab. ")
    name = name.replace("Kota Administrasi ", "Kota ")
    
    return name

supply["region_key"] = supply["region_standard"].apply(
    standardize_region
)

climate["region_key"] = climate["region"].apply(
    standardize_region
)

# Periksa apakah seluruh wilayah berhasil dipasangkan
supply_regions = set(supply["region_key"].unique())
climate_regions = set(climate["region_key"].unique())

print("Wilayah hanya di supply:")
print(sorted(supply_regions - climate_regions))

print("\nWilayah hanya di climate:")
print(sorted(climate_regions - supply_regions))

print("\nJumlah wilayah supply:", len(supply_regions))
print("Jumlah wilayah climate:", len(climate_regions))


Wilayah hanya di supply:
[]

Wilayah hanya di climate:
[]

Jumlah wilayah supply: 38
Jumlah wilayah climate: 38


In [21]:

climate_cols = [
    "region_key",
    "period",
    "rainfall_mm_per_day",
    "rainfall_mm_month",
    "temp_avg_c",
    "temp_max_c",
    "temp_min_c",
    "humidity_pct",
    "solar_mj"
]

# Ambil variabel iklim dari sumber aslinya
climate_selected = climate[climate_cols].copy()

# Hindari duplikasi variabel iklim dari supply
supply_base = supply.drop(
    columns=[
        "rainfall_mm_month",
        "temp_avg_c",
        "humidity_pct",
        "solar_mj"
    ],
    errors="ignore"
)

# Gabungkan
supply_climate = supply_base.merge(
    climate_selected,
    on=["region_key", "period"],
    how="left",
    validate="one_to_one",
    indicator=True
)

print("Hasil merge:")
print(supply_climate["_merge"].value_counts())

print("\nJumlah baris:", len(supply_climate))
print("Jumlah wilayah:", supply_climate["region_key"].nunique())
print("Duplikasi:", supply_climate.duplicated(
    ["region_key", "period"]
).sum())

# Missing value variabel iklim
climate_variables = [
    "rainfall_mm_per_day",
    "rainfall_mm_month",
    "temp_avg_c",
    "temp_max_c",
    "temp_min_c",
    "humidity_pct",
    "solar_mj"
]

print("\nMissing values:")
print(supply_climate[climate_variables].isna().sum())

# Simpan hasil
import os
os.makedirs("outputs", exist_ok=True)

supply_climate.drop(columns=["_merge"]).to_csv(
    "outputs/monthly_supply_climate.csv",
    index=False
)

print("\nDataset tersimpan:")
print("outputs/monthly_supply_climate.csv")


Hasil merge:
_merge
both          2736
left_only        0
right_only       0
Name: count, dtype: int64

Jumlah baris: 2736
Jumlah wilayah: 38
Duplikasi: 0

Missing values:
rainfall_mm_per_day    0
rainfall_mm_month      0
temp_avg_c             0
temp_max_c             0
temp_min_c             0
humidity_pct           0
solar_mj               0
dtype: int64

Dataset tersimpan:
outputs/monthly_supply_climate.csv


In [22]:

import pandas as pd

df = pd.read_csv("outputs/monthly_supply_climate.csv")

print("Ukuran dataset:", df.shape)
print("Jumlah wilayah:", df["region_key"].nunique())
print("Periode:", df["period"].min(), "-", df["period"].max())

print("\nDuplikasi:")
print(df.duplicated(["region_key", "period"]).sum())

climate_vars = [
    "rainfall_mm_month",
    "temp_avg_c",
    "temp_max_c",
    "temp_min_c",
    "humidity_pct",
    "solar_mj"
]

print("\nMissing values:")
print(df[climate_vars].isna().sum())

print("\nMissing values per tahun:")
df["year"] = df["period"].str[:4].astype(int)
print(df.groupby("year")[climate_vars].apply(
    lambda x: x.isna().sum()
))

print("\nStatistik produksi:")
print(df["production_ton"].describe())


Ukuran dataset: (2736, 16)
Jumlah wilayah: 38
Periode: 2020-01 - 2025-12

Duplikasi:
0

Missing values:
rainfall_mm_month    0
temp_avg_c           0
temp_max_c           0
temp_min_c           0
humidity_pct         0
solar_mj             0
dtype: int64

Missing values per tahun:
      rainfall_mm_month  temp_avg_c  temp_max_c  temp_min_c  humidity_pct  \
year                                                                        
2020                  0           0           0           0             0   
2021                  0           0           0           0             0   
2022                  0           0           0           0             0   
2023                  0           0           0           0             0   
2024                  0           0           0           0             0   
2025                  0           0           0           0             0   

      solar_mj  
year            
2020         0  
2021         0  
2022         0  
2023         0  

In [23]:

import pandas as pd

df = pd.read_csv("outputs/monthly_supply_climate.csv")

df["period"] = pd.to_datetime(df["period"])

climate_vars = [
    "rainfall_mm_month",
    "temp_avg_c",
    "temp_max_c",
    "temp_min_c",
    "humidity_pct",
    "solar_mj"
]

df = df.sort_values(["region_key", "period"]).copy()

# Buat lag iklim satu bulan untuk setiap wilayah
for col in climate_vars:
    df[f"{col}_lag1"] = (
        df.groupby("region_key")[col].shift(1)
    )

# Buat lag produksi satu bulan
df["production_lag1"] = (
    df.groupby("region_key")["production_ton"].shift(1)
)

# Simpan dataset fitur
df.to_csv(
    "outputs/monthly_supply_climate_features.csv",
    index=False
)

print("Dataset fitur berhasil dibuat.")
print("Ukuran:", df.shape)

display(
    df[
        [
            "region_key",
            "period",
            "production_ton",
            "production_lag1",
            "rainfall_mm_month_lag1",
            "temp_avg_c_lag1",
            "humidity_pct_lag1"
        ]
    ].head(15)
)


Dataset fitur berhasil dibuat.
Ukuran: (2736, 23)


,region_key,period,production_ton,production_lag1,rainfall_mm_month_lag1,temp_avg_c_lag1,humidity_pct_lag1
0,Kab. Bangkalan,2020-01-01,260.04,NaN,NaN,NaN,NaN
1,Kab. Bangkalan,2020-02-01,1407.42,260.04,345.3,27.96,83.50
2,Kab. Bangkalan,2020-03-01,12772.31,1407.42,435.3,27.46,85.92
3,Kab. Bangkalan,2020-04-01,45623.93,12772.31,284.6,27.86,84.00
4,Kab. Bangkalan,2020-05-01,8237.75,45623.93,329.7,28.23,82.48
5,Kab. Bangkalan,2020-06-01,7228.97,8237.75,212.7,28.34,82.71
6,Kab. Bangkalan,2020-07-01,13756.72,7228.97,48.3,27.67,80.00
7,Kab. Bangkalan,2020-08-01,16627.98,13756.72,27.0,27.73,74.29
8,Kab. Bangkalan,2020-09-01,6626.28,16627.98,25.7,28.68,68.83
9,Kab. Bangkalan,2020-10-01,4220.81,6626.28,13.2,30.19,61.66


In [24]:

import pandas as pd

df = pd.read_csv("outputs/monthly_supply_climate_features.csv")

lag_cols = [
    "rainfall_mm_month_lag1",
    "temp_avg_c_lag1",
    "temp_max_c_lag1",
    "temp_min_c_lag1",
    "humidity_pct_lag1",
    "solar_mj_lag1"
]

print("Missing values fitur lag:")
print(df[lag_cols].isna().sum())

print("\nContoh baris dengan missing:")
display(
    df[df[lag_cols].isna().any(axis=1)][
        ["region_key", "period"] + lag_cols
    ].head(20)
)


Missing values fitur lag:
rainfall_mm_month_lag1    38
temp_avg_c_lag1           38
temp_max_c_lag1           38
temp_min_c_lag1           38
humidity_pct_lag1         38
solar_mj_lag1             38
dtype: int64

Contoh baris dengan missing:


,region_key,period,rainfall_mm_month_lag1,temp_avg_c_lag1,temp_max_c_lag1,temp_min_c_lag1,humidity_pct_lag1,solar_mj_lag1
0,Kab. Bangkalan,2020-01-01,NaN,NaN,NaN,NaN,NaN,NaN
72,Kab. Banyuwangi,2020-01-01,NaN,NaN,NaN,NaN,NaN,NaN
144,Kab. Blitar,2020-01-01,NaN,NaN,NaN,NaN,NaN,NaN
216,Kab. Bojonegoro,2020-01-01,NaN,NaN,NaN,NaN,NaN,NaN
288,Kab. Bondowoso,2020-01-01,NaN,NaN,NaN,NaN,NaN,NaN
360,Kab. Gresik,2020-01-01,NaN,NaN,NaN,NaN,NaN,NaN
432,Kab. Jember,2020-01-01,NaN,NaN,NaN,NaN,NaN,NaN
504,Kab. Jombang,2020-01-01,NaN,NaN,NaN,NaN,NaN,NaN
576,Kab. Kediri,2020-01-01,NaN,NaN,NaN,NaN,NaN,NaN
648,Kab. Lamongan,2020-01-01,NaN,NaN,NaN,NaN,NaN,NaN


In [27]:

import os
import warnings
import numpy as np
import pandas as pd

from statsmodels.tsa.statespace.sarimax import SARIMAX

warnings.filterwarnings("ignore")

# 1. Load dataset
df = pd.read_csv(
    "outputs/monthly_supply_climate_features.csv"
)

df["period"] = pd.to_datetime(df["period"])
df = df.sort_values(["region_key", "period"])

climate_lags = [
    "rainfall_mm_month_lag1",
    "temp_avg_c_lag1",
    "humidity_pct_lag1",
    "solar_mj_lag1"
]

# 2. Pengaturan eksperimen
train_start = pd.Timestamp("2020-02-01")
train_end = pd.Timestamp("2024-12-01")
test_start = pd.Timestamp("2025-01-01")
test_end = pd.Timestamp("2025-12-01")

results = []
failed = []

# 3. Proses per wilayah
for region, group in df.groupby("region_key"):

    try:
        group = (
            group.sort_values("period")
            .set_index("period")
            .asfreq("MS")
        )

        # Training dimulai Februari 2020
        train = group.loc[
            train_start:train_end
        ].copy()

        test = group.loc[
            test_start:test_end
        ].copy()

        # Periksa data
        if len(train) != 59:
            raise ValueError(
                f"Training rows tidak sesuai: {len(train)}"
            )

        if len(test) != 12:
            raise ValueError(
                f"Testing rows tidak sesuai: {len(test)}"
            )

        if train["production_ton"].isna().any():
            raise ValueError("Missing production training")

        if train[climate_lags].isna().any().any():
            raise ValueError("Missing climate training")

        if test[climate_lags].isna().any().any():
            raise ValueError("Missing climate testing")

        # 4. Fit SARIMAX
        model = SARIMAX(
            endog=train["production_ton"],
            exog=train[climate_lags],
            order=(1, 1, 1),
            seasonal_order=(1, 0, 0, 12),
            enforce_stationarity=False,
            enforce_invertibility=False
        )

        fitted = model.fit(
            disp=False,
            maxiter=100
        )

        # 5. Rolling one-step-ahead
        for target_date in test.index:

            exog_target = group.loc[
                [target_date], climate_lags
            ]

            forecast = fitted.get_forecast(
                steps=1,
                exog=exog_target
            )

            forecast_value = float(
                forecast.predicted_mean.iloc[0]
            )

            actual_value = float(
                group.loc[
                    target_date, "production_ton"
                ]
            )

            results.append({
                "region": region,
                "origin": target_date - pd.DateOffset(months=1),
                "target": target_date,
                "horizon": 1,
                "model": "SARIMAX_Climate_Lag1",
                "actual_ton": actual_value,
                "forecast_ton": forecast_value,
                "abs_error": abs(
                    actual_value - forecast_value
                ),
                "squared_error": (
                    actual_value - forecast_value
                ) ** 2
            })

            # Update model dengan data aktual bulan target
            new_endog = group.loc[
                [target_date], "production_ton"
            ]

            fitted = fitted.append(
                endog=new_endog,
                exog=exog_target,
                refit=False
            )

    except Exception as e:
        failed.append({
            "region": region,
            "error": str(e)
        })

# 6. Hasil
predictions = pd.DataFrame(results)

print("Jumlah prediksi:", len(predictions))

if not predictions.empty:
    print(
        "Jumlah wilayah berhasil:",
        predictions["region"].nunique()
    )
else:
    print("Tidak ada prediksi yang berhasil.")

print("Jumlah wilayah gagal:", len(failed))

# 7. Simpan hanya jika ada hasil
output_dir = "outputs/forecasting_benchmark"
os.makedirs(output_dir, exist_ok=True)

if not predictions.empty:
    predictions.to_csv(
        f"{output_dir}/sarimax_climate_h1.csv",
        index=False
    )
    print("Prediksi berhasil disimpan.")

if failed:
    failed_df = pd.DataFrame(failed)
    failed_df.to_csv(
        f"{output_dir}/sarimax_failed_regions.csv",
        index=False
    )

    print("\nDetail kegagalan:")
    display(failed_df)


Jumlah prediksi: 456
Jumlah wilayah berhasil: 38
Jumlah wilayah gagal: 0
Prediksi berhasil disimpan.


In [28]:

if not predictions.empty:

    mae = predictions["abs_error"].mean()

    rmse = np.sqrt(
        predictions["squared_error"].mean()
    )

    print("SARIMAX Climate Lag1")
    print("MAE :", round(mae, 3))
    print("RMSE:", round(rmse, 3))

    print("\nMetrik per wilayah:")

    region_metrics = (
        predictions.groupby("region")
        .agg(
            MAE=("abs_error", "mean"),
            RMSE=("squared_error",
                  lambda x: np.sqrt(x.mean()))
        )
        .reset_index()
    )

    display(region_metrics)
else:
    print(
        "Metrik belum dapat dihitung karena "
        "predictions masih kosong."
    )


SARIMAX Climate Lag1
MAE : 6408.974
RMSE: 11814.232

Metrik per wilayah:


,region,MAE,RMSE
0,Kab. Bangkalan,7909.052684,10678.927589
1,Kab. Banyuwangi,6965.468860,8915.674220
2,Kab. Blitar,3804.045221,6512.664704
3,Kab. Bojonegoro,21162.580157,30938.082600
4,Kab. Bondowoso,3245.525556,4049.703955
5,Kab. Gresik,14222.564802,16854.824020
6,Kab. Jember,10209.087532,12471.737918
7,Kab. Jombang,6164.424909,8402.809779
8,Kab. Kediri,5329.560764,6246.304474
9,Kab. Lamongan,22669.927215,28708.716694


In [29]:

import pandas as pd
import numpy as np

# Benchmark lama
old = pd.read_csv(
    "outputs/forecasting_benchmark/supply_predictions.csv"
)

# Hasil SARIMAX
new = pd.read_csv(
    "outputs/forecasting_benchmark/sarimax_climate_h1.csv"
)

# Samakan format tanggal
for data in [old, new]:
    data["target"] = pd.to_datetime(data["target"])
    data["origin"] = pd.to_datetime(data["origin"])

# Ambil horizon 1 dari benchmark lama
old_h1 = old[
    (old["horizon"] == 1) &
    (old["target"].dt.year == 2025)
].copy()

# Ambil SARIMAX
new_h1 = new.copy()

# Pastikan setiap target dan origin sama
keys = ["region", "origin", "target"]

paired = old_h1.pivot_table(
    index=keys,
    columns="model",
    values="abs_error"
).reset_index()

paired = paired.merge(
    new_h1[keys + ["abs_error"]].rename(
        columns={"abs_error": "SARIMAX"}
    ),
    on=keys,
    how="inner"
)

print("Jumlah observasi berpasangan:", len(paired))

# Hitung metrik hanya dari observasi berpasangan
model_cols = [
    "SeasonalNaive",
    "SARIMA_111_10112",
    "SARIMAX"
]

for model in model_cols:
    if model in paired.columns:
        print(
            model,
            "MAE:",
            round(paired[model].mean(), 3)
        )

print("\nPerbandingan SARIMA vs SARIMAX:")

if "SARIMA_111_10112" in paired.columns:
    paired["improvement_pct"] = (
        (
            paired["SARIMA_111_10112"] -
            paired["SARIMAX"]
        ) /
        paired["SARIMA_111_10112"]
    ) * 100

    print(
        "SARIMAX lebih baik:",
        (paired["improvement_pct"] > 0).sum()
    )

    print(
        "SARIMA lebih baik atau sama:",
        (paired["improvement_pct"] <= 0).sum()
    )

    print(
        "Rata-rata improvement (%):",
        round(paired["improvement_pct"].mean(), 3)
    )

# Simpan hasil
paired.to_csv(
    "outputs/forecasting_benchmark/paired_comparison_h1.csv",
    index=False
)


Jumlah observasi berpasangan: 108
SeasonalNaive MAE: 232.057
SARIMA_111_10112 MAE: 195.908
SARIMAX MAE: 205.186

Perbandingan SARIMA vs SARIMAX:
SARIMAX lebih baik: 51
SARIMA lebih baik atau sama: 57
Rata-rata improvement (%): -44.158


In [30]:

import pandas as pd

old = pd.read_csv(
    "outputs/forecasting_benchmark/supply_predictions.csv"
)

new = pd.read_csv(
    "outputs/forecasting_benchmark/sarimax_climate_h1.csv"
)

old["target"] = pd.to_datetime(old["target"])
old["origin"] = pd.to_datetime(old["origin"])

new["target"] = pd.to_datetime(new["target"])
new["origin"] = pd.to_datetime(new["origin"])

old_h1 = old[
    (old["horizon"] == 1) &
    (old["target"].dt.year == 2025)
].copy()

print("Jumlah baris benchmark h1:", len(old_h1))
print("Jumlah baris SARIMAX:", len(new))

print("\nJumlah wilayah benchmark:")
print(old_h1["region"].nunique())

print("\nJumlah wilayah SARIMAX:")
print(new["region"].nunique())

old_regions = set(old_h1["region"].unique())
new_regions = set(new["region"].unique())

print("\nWilayah hanya di benchmark:")
print(sorted(old_regions - new_regions))

print("\nWilayah hanya di SARIMAX:")
print(sorted(new_regions - old_regions))

print("\nContoh nama wilayah benchmark:")
print(sorted(old_regions)[:10])

print("\nContoh nama wilayah SARIMAX:")
print(sorted(new_regions)[:10])

# Periksa kesamaan pasangan tanggal
old_keys = old_h1[
    ["region", "origin", "target"]
].drop_duplicates()

new_keys = new[
    ["region", "origin", "target"]
].drop_duplicates()

print("\nJumlah key benchmark:", len(old_keys))
print("Jumlah key SARIMAX:", len(new_keys))

print("\nContoh key benchmark:")
display(old_keys.head())

print("\nContoh key SARIMAX:")
display(new_keys.head())


Jumlah baris benchmark h1: 912
Jumlah baris SARIMAX: 456

Jumlah wilayah benchmark:
38

Jumlah wilayah SARIMAX:
38

Wilayah hanya di benchmark:
['Kabupaten Bangkalan', 'Kabupaten Banyuwangi', 'Kabupaten Blitar', 'Kabupaten Bojonegoro', 'Kabupaten Bondowoso', 'Kabupaten Gresik', 'Kabupaten Jember', 'Kabupaten Jombang', 'Kabupaten Kediri', 'Kabupaten Lamongan', 'Kabupaten Lumajang', 'Kabupaten Madiun', 'Kabupaten Magetan', 'Kabupaten Malang', 'Kabupaten Mojokerto', 'Kabupaten Nganjuk', 'Kabupaten Ngawi', 'Kabupaten Pacitan', 'Kabupaten Pamekasan', 'Kabupaten Pasuruan', 'Kabupaten Ponorogo', 'Kabupaten Probolinggo', 'Kabupaten Sampang', 'Kabupaten Sidoarjo', 'Kabupaten Situbondo', 'Kabupaten Sumenep', 'Kabupaten Trenggalek', 'Kabupaten Tuban', 'Kabupaten Tulungagung']

Wilayah hanya di SARIMAX:
['Kab. Bangkalan', 'Kab. Banyuwangi', 'Kab. Blitar', 'Kab. Bojonegoro', 'Kab. Bondowoso', 'Kab. Gresik', 'Kab. Jember', 'Kab. Jombang', 'Kab. Kediri', 'Kab. Lamongan', 'Kab. Lumajang', 'Kab. Madiun

,region,origin,target
0,Kabupaten Bangkalan,2024-12-01,2025-01-01
6,Kabupaten Bangkalan,2025-01-01,2025-02-01
12,Kabupaten Bangkalan,2025-02-01,2025-03-01
18,Kabupaten Bangkalan,2025-03-01,2025-04-01
24,Kabupaten Bangkalan,2025-04-01,2025-05-01



Contoh key SARIMAX:


,region,origin,target
0,Kab. Bangkalan,2024-12-01,2025-01-01
1,Kab. Bangkalan,2025-01-01,2025-02-01
2,Kab. Bangkalan,2025-02-01,2025-03-01
3,Kab. Bangkalan,2025-03-01,2025-04-01
4,Kab. Bangkalan,2025-04-01,2025-05-01


In [31]:

import pandas as pd

old = pd.read_csv(
    "outputs/forecasting_benchmark/supply_predictions.csv"
)

new = pd.read_csv(
    "outputs/forecasting_benchmark/sarimax_climate_h1.csv"
)

for df in [old, new]:
    df["target"] = pd.to_datetime(df["target"])
    df["origin"] = pd.to_datetime(df["origin"])

    # Samakan format nama wilayah
    df["region_key"] = (
        df["region"]
        .str.replace("Kabupaten ", "Kab. ", regex=False)
        .str.replace("Kota ", "Kota ", regex=False)
        .str.strip()
    )

old_h1 = old[
    (old["horizon"] == 1) &
    (old["target"].dt.year == 2025)
].copy()

new_h1 = new.copy()

keys = ["region_key", "origin", "target"]

# Pisahkan error SARIMA dan Seasonal Naive
old_pivot = old_h1.pivot_table(
    index=keys,
    columns="model",
    values="abs_error"
).reset_index()

# Gabungkan dengan SARIMAX
paired = old_pivot.merge(
    new_h1[keys + ["abs_error"]].rename(
        columns={"abs_error": "SARIMAX"}
    ),
    on=keys,
    how="inner"
)

print("Jumlah observasi berpasangan:", len(paired))

print("\nMAE masing-masing model:")
for model in [
    "SeasonalNaive",
    "SARIMA_111_10112",
    "SARIMAX"
]:
    print(model, round(paired[model].mean(), 3))

print("\nRMSE masing-masing model:")
for model in [
    "SeasonalNaive",
    "SARIMA_111_10112",
    "SARIMAX"
]:
    print(
        model,
        round((paired[model] ** 2).mean() ** 0.5, 3)
    )

# Perbandingan SARIMA vs SARIMAX
paired["improvement_pct"] = (
    (paired["SARIMA_111_10112"] - paired["SARIMAX"])
    / paired["SARIMA_111_10112"]
) * 100

print("\nPerbandingan SARIMA vs SARIMAX:")
print(
    "SARIMAX lebih baik:",
    (paired["SARIMAX"] < paired["SARIMA_111_10112"]).sum()
)
print(
    "SARIMA lebih baik atau sama:",
    (paired["SARIMA_111_10112"] <= paired["SARIMAX"]).sum()
)
print(
    "Rata-rata improvement (%):",
    round(paired["improvement_pct"].mean(), 3)
)
print(
    "Median improvement (%):",
    round(paired["improvement_pct"].median(), 3)
)

paired.to_csv(
    "outputs/forecasting_benchmark/paired_comparison_h1.csv",
    index=False
)


Jumlah observasi berpasangan: 456

MAE masing-masing model:
SeasonalNaive 5037.511
SARIMA_111_10112 5016.056
SARIMAX 6408.974

RMSE masing-masing model:
SeasonalNaive 10189.728
SARIMA_111_10112 10217.657
SARIMAX 11814.232

Perbandingan SARIMA vs SARIMAX:
SARIMAX lebih baik: 180
SARIMA lebih baik atau sama: 276
Rata-rata improvement (%): -169.551
Median improvement (%): -14.165


In [32]:

import pandas as pd
import numpy as np

paired = pd.read_csv(
    "outputs/forecasting_benchmark/paired_comparison_h1.csv"
)

sarima = "SARIMA_111_10112"

# Selisih error positif berarti SARIMAX lebih buruk
paired["error_difference"] = (
    paired["SARIMAX"] - paired[sarima]
)

# Ringkasan per wilayah
region_result = (
    paired.groupby("region_key")
    .agg(
        MAE_SARIMA=(sarima, "mean"),
        MAE_SARIMAX=("SARIMAX", "mean"),
        mean_error_difference=("error_difference", "mean"),
        n=("error_difference", "count")
    )
    .reset_index()
)

region_result["improvement_pct"] = (
    (region_result["MAE_SARIMA"] -
     region_result["MAE_SARIMAX"])
    / region_result["MAE_SARIMA"]
) * 100

print("Jumlah wilayah SARIMAX lebih baik:",
      (region_result["improvement_pct"] > 0).sum())

print("\n5 wilayah dengan peningkatan terbaik:")
display(
    region_result.sort_values(
        "improvement_pct", ascending=False
    ).head(5)
)

print("\n5 wilayah dengan penurunan terbesar:")
display(
    region_result.sort_values(
        "improvement_pct"
    ).head(5)
)

print("\n10 observasi dengan kenaikan error terbesar:")
display(
    paired.sort_values(
        "error_difference", ascending=False
    )[
        ["region_key", "origin", "target",
         sarima, "SARIMAX", "error_difference"]
    ].head(10)
)


Jumlah wilayah SARIMAX lebih baik: 6

5 wilayah dengan peningkatan terbaik:


,region_key,MAE_SARIMA,MAE_SARIMAX,mean_error_difference,n,improvement_pct
35,Kota Pasuruan,284.776800,256.124797,-28.652003,12,10.061214
10,Kab. Lumajang,2892.543266,2646.662029,-245.881237,12,8.500521
29,Kota Batu,93.909808,89.098802,-4.811006,12,5.123007
7,Kab. Jombang,6439.165078,6164.424909,-274.740169,12,4.266705
14,Kab. Mojokerto,4242.807739,4075.155813,-167.651926,12,3.951438



5 wilayah dengan penurunan terbesar:


,region_key,MAE_SARIMA,MAE_SARIMAX,mean_error_difference,n,improvement_pct
6,Kab. Jember,5009.812669,10209.087532,5199.274864,12,-103.781822
8,Kab. Kediri,2710.116454,5329.560764,2619.444310,12,-96.654308
11,Kab. Madiun,4964.960275,9406.639073,4441.678798,12,-89.460510
27,Kab. Tuban,8542.139304,14908.503549,6366.364246,12,-74.528921
18,Kab. Pamekasan,5536.412396,8900.135722,3363.723326,12,-60.756372



10 observasi dengan kenaikan error terbesar:


,region_key,origin,target,SARIMA_111_10112,SARIMAX,error_difference
197,Kab. Ngawi,2025-05-01,2025-06-01,11093.832825,40808.795436,29714.962612
119,Kab. Lamongan,2025-11-01,2025-12-01,9293.450834,38105.648476,28812.197642
112,Kab. Lamongan,2025-04-01,2025-05-01,10180.503477,37100.623916,26920.120440
108,Kab. Lamongan,2024-12-01,2025-01-01,10190.676406,36778.277057,26587.600651
133,Kab. Madiun,2025-01-01,2025-02-01,2868.193618,23990.470584,21122.276966
36,Kab. Bojonegoro,2024-12-01,2025-01-01,2025.558427,22509.824850,20484.266423
335,Kab. Tuban,2025-11-01,2025-12-01,919.336925,19393.808156,18474.471230
328,Kab. Tuban,2025-04-01,2025-05-01,18920.045009,37224.067761,18304.022752
194,Kab. Ngawi,2025-02-01,2025-03-01,3230.824129,21373.231321,18142.407192
43,Kab. Bojonegoro,2025-07-01,2025-08-01,7023.888229,23789.128490,16765.240261


In [34]:

import pandas as pd

df = pd.read_csv(
    "outputs/monthly_supply_climate_features.csv"
)

print("Ukuran dataset:", df.shape)
print("\nDaftar kolom:")
print(df.columns.tolist())

print("\nLima baris pertama:")
display(df.head())


Ukuran dataset: (2736, 23)

Daftar kolom:
['region_id', 'region_standard', 'year', 'month', 'period', 'production_ton', 'source_type', 'production_unit', 'region_key', 'rainfall_mm_per_day', 'rainfall_mm_month', 'temp_avg_c', 'temp_max_c', 'temp_min_c', 'humidity_pct', 'solar_mj', 'rainfall_mm_month_lag1', 'temp_avg_c_lag1', 'temp_max_c_lag1', 'temp_min_c_lag1', 'humidity_pct_lag1', 'solar_mj_lag1', 'production_lag1']

Lima baris pertama:


,region_id,region_standard,year,month,period,production_ton,source_type,production_unit,region_key,rainfall_mm_per_day,...,temp_min_c,humidity_pct,solar_mj,rainfall_mm_month_lag1,temp_avg_c_lag1,temp_max_c_lag1,temp_min_c_lag1,humidity_pct_lag1,solar_mj_lag1,production_lag1
0,KAB_BANGKALAN,Kabupaten Bangkalan,2020,1,2020-01-01,260.04,historical_observation,ton,Kab. Bangkalan,11.14,...,25.02,83.50,18.75,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,KAB_BANGKALAN,Kabupaten Bangkalan,2020,2,2020-02-01,1407.42,historical_observation,ton,Kab. Bangkalan,15.01,...,24.95,85.92,18.56,345.3,27.96,31.67,25.02,83.50,18.75,260.04
2,KAB_BANGKALAN,Kabupaten Bangkalan,2020,3,2020-03-01,12772.31,historical_observation,ton,Kab. Bangkalan,9.18,...,25.30,84.00,18.56,435.3,27.46,30.71,24.95,85.92,18.56,1407.42
3,KAB_BANGKALAN,Kabupaten Bangkalan,2020,4,2020-04-01,45623.93,historical_observation,ton,Kab. Bangkalan,10.99,...,25.31,82.48,17.77,284.6,27.86,30.77,25.30,84.00,18.56,12772.31
4,KAB_BANGKALAN,Kabupaten Bangkalan,2020,5,2020-05-01,8237.75,historical_observation,ton,Kab. Bangkalan,6.86,...,26.17,82.71,17.16,329.7,28.23,30.99,25.31,82.48,17.77,45623.93


In [35]:

import pandas as pd

df = pd.read_csv(
    "outputs/monthly_supply_climate_features.csv"
)

# Deteksi kolom
region_candidates = ["region", "region_standard", "region_key"]
period_candidates = ["period", "date", "month"]
production_candidates = [
    "produksi_beras",
    "production_ton",
    "supply_ton"
]

def find_column(candidates, columns):
    for col in candidates:
        if col in columns:
            return col
    return None

region_col = find_column(region_candidates, df.columns)
period_col = find_column(period_candidates, df.columns)
production_col = find_column(production_candidates, df.columns)

print("Kolom wilayah:", region_col)
print("Kolom periode:", period_col)
print("Kolom produksi:", production_col)

if not all([region_col, period_col, production_col]):
    raise ValueError(
        "Kolom tidak ditemukan. Periksa daftar kolom dataset."
    )

df[period_col] = pd.to_datetime(df[period_col])

# Fokus pada Lamongan
lamongan = df[
    df[region_col].str.contains(
        "Lamongan", case=False, na=False
    )
].copy()

lamongan = lamongan.sort_values(period_col)

lamongan = lamongan[
    (lamongan[period_col] >= "2024-01-01") &
    (lamongan[period_col] <= "2025-12-01")
]

# Pilih kolom yang memang tersedia
desired_cols = [
    period_col,
    production_col,
    "rainfall_mm_month",
    "temp_avg_c",
    "humidity_pct",
    "solar_mj",
    "rainfall_mm_month_lag1",
    "temp_avg_c_lag1",
    "humidity_pct_lag1",
    "solar_mj_lag1"
]

available_cols = [
    col for col in desired_cols if col in lamongan.columns
]

print("\nJumlah observasi Lamongan:", len(lamongan))
display(lamongan[available_cols])


Kolom wilayah: region_standard
Kolom periode: period
Kolom produksi: production_ton

Jumlah observasi Lamongan: 24


,period,production_ton,rainfall_mm_month,temp_avg_c,humidity_pct,solar_mj,rainfall_mm_month_lag1,temp_avg_c_lag1,humidity_pct_lag1,solar_mj_lag1
696,2024-01-01,2300.19,293.9,28.08,84.30,17.44,768.8,28.77,80.32,19.29
697,2024-02-01,4264.98,388.0,27.97,86.03,17.89,293.9,28.08,84.30,17.44
698,2024-03-01,102923.38,205.5,28.22,85.50,16.88,388.0,27.97,86.03,17.89
699,2024-04-01,99868.18,264.0,28.77,83.65,16.94,205.5,28.22,85.50,16.88
700,2024-05-01,18724.35,18.9,29.03,78.57,18.40,264.0,28.77,83.65,16.94
701,2024-06-01,40074.65,69.6,28.84,75.89,17.07,18.9,29.03,78.57,18.40
702,2024-07-01,48059.03,27.3,28.48,70.03,18.98,69.6,28.84,75.89,17.07
703,2024-08-01,52036.08,3.7,29.54,61.11,21.07,27.3,28.48,70.03,18.98
704,2024-09-01,64211.69,39.3,30.08,64.23,20.80,3.7,29.54,61.11,21.07
705,2024-10-01,12600.99,35.0,31.05,62.27,21.36,39.3,30.08,64.23,20.80


In [36]:

# Ringkasan statistik dan nilai ekstrem
print("Produksi tertinggi:")
display(
    lamongan.nlargest(5, production_col)[
        [period_col, production_col]
    ]
)

print("\nProduksi terendah:")
display(
    lamongan.nsmallest(5, production_col)[
        [period_col, production_col]
    ]
)

print("\nData yang memiliki lag iklim kosong:")
display(
    lamongan[
        lamongan["rainfall_mm_month_lag1"].isna()
    ][available_cols]
)


Produksi tertinggi:


,period,production_ton
710,2025-03-01,144391.63
698,2024-03-01,102923.38
699,2024-04-01,99868.18
704,2024-09-01,64211.69
713,2025-06-01,57222.24



Produksi terendah:


,period,production_ton
706,2024-11-01,864.74
708,2025-01-01,948.55
696,2024-01-01,2300.19
707,2024-12-01,2317.62
719,2025-12-01,2419.62



Data yang memiliki lag iklim kosong:


,period,production_ton,rainfall_mm_month,temp_avg_c,humidity_pct,solar_mj,rainfall_mm_month_lag1,temp_avg_c_lag1,humidity_pct_lag1,solar_mj_lag1


In [38]:

import pandas as pd

old = pd.read_csv(
    "outputs/forecasting_benchmark/supply_predictions.csv"
)

new = pd.read_csv(
    "outputs/forecasting_benchmark/sarimax_climate_h1.csv"
)

# Standardisasi nama wilayah dan tanggal
for df in [old, new]:
    df["region_key"] = (
        df["region"]
        .str.replace("Kabupaten ", "Kab. ", regex=False)
        .str.strip()
    )
    df["origin"] = pd.to_datetime(df["origin"])
    df["target"] = pd.to_datetime(df["target"])

# Filter Lamongan, horizon 1, tahun target 2025
sarima = old[
    (old["region_key"] == "Kab. Lamongan") &
    (old["horizon"] == 1) &
    (old["target"].dt.year == 2025) &
    (old["model"] == "SARIMA_111_10112")
].copy()

sarimax = new[
    (new["region_key"] == "Kab. Lamongan") &
    (new["target"].dt.year == 2025)
].copy()

keys = ["region_key", "origin", "target"]

# Periksa jumlah key unik
print("SARIMA key unik:", sarima[keys].drop_duplicates().shape[0])
print("SARIMAX key unik:", sarimax[keys].drop_duplicates().shape[0])

# Merge dengan ketiga key
comparison = sarima[
    keys + ["actual_ton", "forecast_ton", "abs_error"]
].rename(columns={
    "forecast_ton": "forecast_sarima",
    "abs_error": "error_sarima"
}).merge(
    sarimax[
        keys + ["actual_ton", "forecast_ton", "abs_error"]
    ].rename(columns={
        "actual_ton": "actual_sarimax",
        "forecast_ton": "forecast_sarimax",
        "abs_error": "error_sarimax"
    }),
    on=keys,
    how="inner",
    validate="one_to_one"
)

print("\nJumlah observasi berpasangan:", len(comparison))

# Pastikan nilai aktual konsisten
print(
    "Aktual konsisten:",
    (
        comparison["actual_ton"] -
        comparison["actual_sarimax"]
    ).abs().max() < 1e-6
)

comparison["error_difference"] = (
    comparison["error_sarimax"] -
    comparison["error_sarima"]
)

display(
    comparison.sort_values("target").round(2)
)


SARIMA key unik: 12
SARIMAX key unik: 12

Jumlah observasi berpasangan: 12
Aktual konsisten: True


,region_key,origin,target,actual_ton,forecast_sarima,error_sarima,actual_sarimax,forecast_sarimax,error_sarimax,error_difference
0,Kab. Lamongan,2024-12-01,2025-01-01,948.55,11139.23,10190.68,948.55,37726.83,36778.28,26587.60
1,Kab. Lamongan,2025-01-01,2025-02-01,38222.75,12179.93,26042.82,38222.75,47084.05,8861.30,-17181.52
2,Kab. Lamongan,2025-02-01,2025-03-01,144391.63,83269.42,61122.21,144391.63,78221.37,66170.26,5048.04
3,Kab. Lamongan,2025-03-01,2025-04-01,44043.33,86134.17,42090.84,44043.33,59487.68,15444.35,-26646.49
4,Kab. Lamongan,2025-04-01,2025-05-01,15685.41,25865.91,10180.50,15685.41,52786.03,37100.62,26920.12
5,Kab. Lamongan,2025-05-01,2025-06-01,57222.24,39426.91,17795.33,57222.24,48590.64,8631.60,-9163.73
6,Kab. Lamongan,2025-06-01,2025-07-01,54055.62,44390.35,9665.27,54055.62,50843.78,3211.84,-6453.43
7,Kab. Lamongan,2025-07-01,2025-08-01,48016.77,48166.21,149.44,48016.77,57401.70,9384.93,9235.49
8,Kab. Lamongan,2025-08-01,2025-09-01,35430.26,57779.14,22348.88,35430.26,57800.34,22370.08,21.21
9,Kab. Lamongan,2025-09-01,2025-10-01,38003.95,19826.60,18177.35,38003.95,26194.85,11809.10,-6368.25


In [39]:

# Cari bagian kode benchmark SARIMA yang digunakan sebelumnya.
# Tampilkan informasi berikut dari kode tersebut:

print("Model SARIMA yang digunakan:")
print("order = (1, 1, 1)")
print("seasonal_order = (1, 0, 0, 12)")

print("\nPeriksa kode benchmark:")
print("1. Apakah model di-fit ulang pada setiap forecast origin?")
print("2. Apakah observasi aktual ditambahkan setelah setiap prediksi?")
print("3. Apakah parameter model diestimasi ulang atau tetap?")


Model SARIMA yang digunakan:
order = (1, 1, 1)
seasonal_order = (1, 0, 0, 12)

Periksa kode benchmark:
1. Apakah model di-fit ulang pada setiap forecast origin?
2. Apakah observasi aktual ditambahkan setelah setiap prediksi?
3. Apakah parameter model diestimasi ulang atau tetap?
